# Import package

In [1]:
from bs4 import BeautifulSoup as bs     # parsing
from urllib.request import urlopen      # scrapping
import pandas as pd                     # exporting
from tenacity import retry, stop_after_attempt
import http.client

In [2]:
from io import StringIO
from pdfminer.converter import TextConverter
from pdfminer.layout import LAParams
from pdfminer.pdfinterp import PDFResourceManager, PDFPageInterpreter
from pdfminer.pdfpage import PDFPage
import os
import logging
from tqdm import tqdm
import csv
import re

# Variables

In [7]:
city = re.compile('[\w\s]{1,20}[,] \w{1,}?[\d\W]{1,11} [\w\s]{1,3}')
# zip_code = re.compile(r'\b\d{5}(?:[-\s]\d{4})?\b')
date = re.compile('(?:(?<!\:)(?<!\:\d)[0-3]?\d(?:st|nd|rd|th)?\s+(?:of\s+)?(?:jan\.?|january|feb\.?|february|mar\.?|march|apr\.?|april|may|jun\.?|june|jul\.?|july|aug\.?|august|sep\.?|september|oct\.?|october|nov\.?|november|dec\.?|december)|(?:jan\.?|january|feb\.?|february|mar\.?|march|apr\.?|april|may|jun\.?|june|jul\.?|july|aug\.?|august|sep\.?|september|oct\.?|october|nov\.?|november|dec\.?|december)\s+(?<!\:)(?<!\:\d)[0-3]?\d(?:st|nd|rd|th)?)(?:\,)?\s*(?:\d{4})?|[0-3]?\d[-\./][0-3]?\d[-\./]\d{2,4}', re.IGNORECASE)
number = re.compile('\d{1,20}')
title = ['PRESIDENT', 'VICE PRESIDENT', 'VICE PRESIDENT, TAX', 'TREASURER', 'ASSISTANT TREASURER', 'ASST. TREASURER', 'ASSISTANT CLERK', 'CLERK',
         'SECRETARY', 'ASSISTANT SECRETARY', 'CORPORATE SECRETARY', 'GENERAL SECRETARY', 'DIRECTOR', 'SECRETARY & DIRECTOR', 'MEDICAL DIRECTOR',
         'ASST TREAS/SECY', 'EDUCATION DIRECTOR', 'CO-WARDEN', 'VICE PRESIDENT FOOTBALL', 'VICE PRESIDENT CHEER', 'REGISTRAR', 'PUBLIC RELATIONS',
         'SENIOR PASTOR', 'ASSISTANT PASTOR', 'CHURCH LEADER', 'COLLECTOR', 'MODERATOR', 'CHAIR', 'GENERAL MEMBER', 'CEO', 'CFO',
         'GENERAL EVANGELIST', 'EXECUTIVE MINISTER', 'CORP CONTROLLER', 'CORPORATE CONTROLLER', 'OFFICER', 'CHAIRMAN & CEO', 'MANAGER',
         'PRESIDENT & CHIEF OPERATING', 'OPERATIONS', 'EDUCATION', 'CHAIRMAN', 'ICE DIRECTOR', 'CHIEF FINANCIAL OFFICER', 'DIRECTOR OF FINANCE',
         'CHIEF EXECUTIVE OFFICER', 'CHIEF OPERATING OFFICER', 'CHIEF COMPLIANCE OFFICER', 'IMMEDIATE PAST CHAIR', 'CHAIR-ELECT',
         'DIRECTOR-CHAIRMAN', 'CFO-TREASURER', 'EVP-COO', 'SENIOR MINISTER', 'MISSIONARY', 'ARCHIVIST']

In [8]:
address_1 = re.compile('(?:ONE|\d{1,4})?[\w\s\W]{1,20}(?:STREET|ST|SRTEET|STR|AVENUE|AVE|ROAD|RD|HIGHWAY|HWY|SQUARE|SQ|TRAIL|TRL|DRIVE|DR|COURT|CT|PARK|PARKWAY|PKWY|CIRCLE|CIR|BOULEVARD|BLVD|TERRACE|TERR|TER|LANE|LN|HL|PATH|WAY|PLACE|BEACH|ROUTE)?\.?\W?(?=\s|$)', re.IGNORECASE)
address_2 = re.compile('(?:ONE|\d{1,4})[\w\s\W]{1,20}(?:STREET|ST|SRTEET|STR|AVENUE|AVE|ROAD|RD|HIGHWAY|HWY|SQUARE|SQ|TRAIL|TRL|DRIVE|DR|COURT|CT|PARK|PARKWAY|PKWY|CIRCLE|CIR|BOULEVARD|BLVD|TERRACE|TERR|TER|LANE|LN|HL|PATH|WAY|PLACE|BEACH|ROUTE)?\.?\W?(?=\s|$)', re.IGNORECASE)

# Util functions

In [3]:
def process_pdf(rsrcmgr, device, fp, pagenos=None, maxpages=0, password='', caching=True, check_extractable=True):
    interpreter = PDFPageInterpreter(rsrcmgr, device)
    for page in PDFPage.get_pages(fp, pagenos, maxpages=maxpages, password=password,
                                  caching=caching, check_extractable=check_extractable):
        interpreter.process_page(page)
    return

In [4]:
def read_pdf(pdf):
    rsrcmgr = PDFResourceManager()
    retstr = StringIO()
    laparams = LAParams()
    device = TextConverter(rsrcmgr, retstr, laparams=laparams)
    process_pdf(rsrcmgr, device, pdf)
    device.close()
    content = retstr.getvalue()
    retstr.close()
    lines = str(content).split("\n")
    return lines

In [5]:
def write_csv(content, path):
    with open(path, 'w') as f:
        w = csv.writer(f)
        w.writerows(content.items())

In [6]:
def handle(rst, phase):
    '''
    param rst: str, the object representing the info extracted from the pdf, usually the result from function read_pdf
    param phase: int, the phase protocol, to signal which regex compiler should be in use
    '''
#     print(rst)
    idnum = ''
    exactName = ''
    jurisdiction = ''
    registeredName = ''
    registeredAddress = ''
    principalAddress = ''
    directors_info = []
    business = ''
    address = None
    
    if phase == 1:
        address = address_1
    elif phase == 2:
        address = address_2
    
    for i in range(len(rst)):
        if 'Identification Number' in rst[i]:
            idnum = rst[i].split(': ')[1].strip()
        
        if 'Exact name' in rst[i] or 'exact name' in rst[i]:
            exactName = rst[i].split(': ')[1].lstrip()
                
            count = i + 1
            tmp = rst[count]
            while 'Jurisdiction of Incorporation' not in tmp and 'registered office' not in tmp and \
                  'principal office' not in tmp:
                exactName += tmp.strip()
                count += 1
                tmp = rst[count]
            
            exactName = exactName.split('2.')[0].strip()
            
        if 'Jurisdiction of Incorporation' in rst[i]:
            state = ''
            country = ''
            count = i
            tmp = rst[count]
            while 'registered office' not in tmp and 'principal office' not in tmp:
                if 'State' in tmp:
                    state = tmp.split('State: ')[1][0:2]
                if 'Country' in tmp:
                    country = tmp.split('Country: ')[1].strip()
                count += 1
                tmp = rst[count]

            if len(country) != 0:
                jurisdiction = state + ', ' + country
            else:
                jurisdiction = state
            
        if 'corporation registered office' in rst[i]:
            registeredStreetList = []
            registeredStreet = ''
            registeredCity = ''
            registeredState = ''
            registeredZip = ''
            registeredCountry = ''
            count = i + 1
            tmp = rst[count]
            while 'principal office' not in tmp and 'directors' not in tmp:
                if (address.search(tmp) != None) and (tmp.isupper()):
                    if ',' in tmp and tmp.strip().split(',')[1] != '':
                        registeredStreetList.append(tmp.strip())
                    else:
                        registeredStreetList.append(tmp.split(',')[0].strip())
                elif registeredName == '':
                    if tmp.isupper():
                        registeredName = tmp.strip()
                elif tmp.isupper() and tmp.replace(' ', '').replace('.', '').isalpha():
                    registeredCity = tmp.strip()
                    
                if 'State' in tmp:
                    try:
                        registeredState = tmp.split('State: ')[1].strip()
                    except:
                        registeredState = ''
                if 'Zip' in tmp:
                    try:
                        registeredZip = re.findall(r'\b\d{5}(?:[-\s]\d{4})?\b', tmp)[0]
                    except:
                        registeredZip = ''
                if 'Country' in tmp:
                    registeredCountry = tmp.split('Country: ')[1].strip()
                    
                count += 1
                tmp = rst[count]
            
            for i in range(len(registeredStreetList)):
                registeredStreet += registeredStreetList[i]
                registeredStreet += ', '
            
            registeredAddress = registeredStreet + registeredCity + ', ' + registeredState + \
                                ' ' + registeredZip + ' ' + registeredCountry
            
        if 'principal office' in rst[i]:
            principalStreetList = []
            principalStreet = ''
            principalCity = ''
            principalState = ''
            principalZip = ''
            principalCountry = ''
            count = i + 1
            tmp = rst[count]
            while 'directors' not in tmp:
                if tmp.isupper() and '3.' not in tmp:
                    if ',' in tmp and tmp.strip().split(',')[1] != '':
                        principalStreetList.append(tmp.strip())
                    else:
                        principalStreetList.append(tmp.split(',')[0].strip())
                
                if 'State' in tmp:
                    try:
                        principalState = tmp.split(': ')[1].strip()
                    except:
                        principalState = ''
                if 'Zip' in tmp:
                    try:
                        principalZip = re.findall(r'\b\d{5}(?:[-\s]\d{4})?\b', tmp)[0]
                    except:
                        principalZip = ''
                if 'Country' in tmp:
                    principalCountry = tmp.split(': ')[1].strip()
                
                count += 1
                tmp = rst[count]
            
            for i in range(len(principalStreetList)):
                principalStreet += principalStreetList[i]
                principalStreet += ', '
            
            principalAddress = principalStreet + principalState + \
                               ' ' + principalZip + ' ' + principalCountry
            
        if 'Title' in rst[i]:
            directorsTitle = []
            directorsName = []
            directorsStreet = []
            directorsCity = []
            count = i + 1
            tmp = rst[count]
            while 'Briefly describe' not in tmp and 'cemetery corporation' not in tmp and 'undersigned' not in tmp:
                # sometimes the title name varies, we can just check if it contains any one in the title list
                if tmp.replace('\xa0', '').strip() in title or \
                bool([t for t in title if (t in tmp.replace('\xa0', '').strip())]): 
                    directorsTitle.append(tmp.strip())
                elif ((city.search(tmp) != None) and (tmp.isupper())) or ('CITY' in tmp):
                    directorsCity.append(tmp.strip())
                
                # usually the column of name is on the left of the address, so this one step should go first
                elif (tmp.isupper()) and (len(tmp.split()) > 1) and \
                     (re.sub(r'[^\w\s]','',tmp).replace('\xa0', '').replace(' ', '').isalpha()):
                    if address.search(tmp) == None:
                        directorsName.append(tmp.strip())
                    elif ',' in tmp and tmp.strip().split(',')[1] != '':
                        directorsStreet.append(tmp.strip())
                    else:
                        directorsStreet.append(tmp.split(',')[0].strip())
                        
                elif ((address.search(tmp) != None) and (tmp.isupper())) or ('PO BOX' in tmp) or (tmp.strip() == 'SAME'):
                    if ',' in tmp and tmp.strip().split(',')[1] != '':
                        directorsStreet.append(tmp.strip())
                    else:
                        directorsStreet.append(tmp.split(',')[0].strip())
                    
                count += 1
                tmp = rst[count]
            
            if len(directorsCity) == 0:
                while 'Check' not in tmp:
                    if ((city.search(tmp) != None) and (tmp.isupper())) or ('CITY' in tmp):
                        directorsCity.append(tmp.strip())
                    elif ((address.search(tmp) != None) and (tmp.isupper())) or ('PO BOX' in tmp):
                        directorsStreet.append(tmp.split(',')[0].strip())
                        
                    count += 1
                    tmp = rst[count]
                
            
            directors_info = [[] for _ in range(len(directorsTitle))]     
        
#             print(directorsTitle)
#             print(directorsName)
#             print(directorsCity)
#             print(directorsStreet)

            if len(directorsTitle) == len(directorsName) or len(directorsName) == len(directorsStreet[i]) or \
                len(directorsStreet[i]) == len(directorsCity[i]):
                for i in range(len(directorsTitle)):
#                     print(i)
    #                 print('info', directors_info[i])
    #                 print('title', directorsTitle[i])
    #                 print('name', directorsName[i])
    #                 print('street', directorsStreet[i])
    #                 print('city', directorsCity[i])
                    try:
                        directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i] + ', ' + \
                                                 directorsStreet[i] + ', ' + directorsCity[i])
                    except:
                        directors_info[i].append(directorsTitle[i] + ', ' + directorsName[i])
                
        if 'The general character' in rst[i] or 'Briefly describe' in rst[i]:
            count = i + 1
#             tmp = rst[count]
            while 'cid' in tmp or tmp == ' ' or tmp == '' or 'service, the service to be rendered:' in tmp or \
                address.search(tmp) != None or city.search(tmp) != None or tmp.isupper() == False:
                count += 1
                tmp = rst[count]
                break
            business = tmp.strip()
                    
    rec = {

        'ID Number': format(int(idnum), '09d') if len(idnum) <= 9  and len(idnum) else None, # fixed length = 9

        'Business Name': exactName,

        'Jurisdiction': jurisdiction,
        
        'Registered Office Name': registeredName,
        
        'Registered Office Address': registeredAddress,
        
        'Principal Office Address': principalAddress,
        
        'Directors': directors_info,
        
        'Business': business
        
        }
    
    return rec

## Extract info using the regex compiler "address_1"

In [ ]:
logging.propagate = False
logging.getLogger().setLevel(logging.ERROR)

path = '.\PDF_data\PDF'
dirs = os.listdir(path)
# record the scanned files' number
scanned = []
# business = {}
results = [] 
for file in tqdm(dirs):
    pdf = path + '/' + file
#     print('the current file is', pdf)
    with open(pdf, 'rb') as f:
        try:
            rst = read_pdf(f)
            if len(rst) == 1:
                # print('This is a scanned version')
                scanned.append(file)
                continue
        except:
            continue    
        if len(rst) == 1:
            continue
        
        try:
            info = handle(rst, 1)
            # use the file name as the ID number
            if info['ID Number'] is None:
                info['ID Number'] = format(int(file[:-4]), '09d')
            results.append(info)
        except IndexError:
            # print('list index out of range')
            continue
        except ValueError:
            info = handle(rst)
            # use the file name as the ID number
            if info['ID Number'] is None:
                info['ID Number'] = (file[:-4])
            results.append(info)
            continue

resultsDF = pd.DataFrame(results)
resultsDF.to_csv('~/Desktop/sec_results.csv')

## save the scanned files' names to local machine

In [9]:
with open('PDF.txt', 'w') as f:
    for item in scanned:
        f.write('%s\n' % item)

NameError: name 'scanned' is not defined

## Move the scanned files to another folder

In [ ]:
import shutil

try:
    for file in tqdm(open("PDF.txt", "r").read().split('\n')):
        shutil.move(path + '/' + file, './../Desktop/PDF' + '/' + file)
except:
        for file in tqdm(scanned):
        shutil.move(path + '/' + file, './../Desktop/PDF' + '/' + file)

## extract the names of files that have not been extracted

In [ ]:
all = pd.DataFrame({'ID Number': dirs)})
# 
try:
    scraped = pd.concat([pd.DataFrame({'ID Number': open("PDF.txt", "r").read().split('\n')}), 
               pd.DataFrame({'ID Number': pd.read_csv('~/Desktop/sec_results2.csv')['ID Number']})])
except:
    print('PDF.txt does not exist')
    scraped = pd.concat([pd.DataFrame({'ID Number': scanned}),
               pd.DataFrame({'ID Number': pd.read_csv('~/Desktop/sec_results2.csv')['ID Number']})])
unscraped = all.loc[~all['ID Number'].isin(scraped['ID Number'])]

## extract the info in the unextracted files using "address_2"

In [ ]:
logging.propagate = False
logging.getLogger().setLevel(logging.ERROR)

path = '.\PDF_data\PDF'
dirs = os.listdir(path)
scanned = []
# business = {}
results = [] 
for file in tqdm(unscraped['ID Number'].values.tolist()):
    pdf = path + '/' + file
    with open(pdf, 'rb') as f:
        try:
            rst = read_pdf(f)
            if len(rst) == 1:
                # print('This is a scanned version')
                scanned.append(file)
                continue
        except:
            continue    
        if len(rst) == 1:
            continue
        
        try:
            info = handle(rst, 2)
            # use the file name as the ID number
            if info['ID Number'] is None:
                info['ID Number'] = format(int(file[:-4]), '09d')
            results.append(info)
        except IndexError:
            # print('list index out of range')
            continue
        except ValueError:
            info = handle(rst)
            # use the file name as the ID number
            if info['ID Number'] is None:
                info['ID Number'] = (file[:-4])
            results.append(info)
            continue
        
#         try:
#             a = business[info[0]]
#         except KeyError:
#             if info[0] == '' or info[1] == '':
#                 continue
#         business[info[0]] = info[1]
            
# write_csv(business, '/Users/apple/Desktop/result.csv')

resultsDF = pd.DataFrame(results)
resultsDF.to_csv('~/Desktop/sec_results2.csv')

# Polishment
----
There are some defects after 2-phase extraction, I suggest to polish accordingly and then concatenate them. Here is the example of my operations towards the folder "PDF".

In [ ]:
result1 = pd.read_csv('~/Desktop/sec_results.csv',index_col=0) # the csv file generated in the first phase
result2 = pd.read_csv('~/Desktop/sec_results2.csv',index_col=0) # the csv file generated in the second phase

In [ ]:
# drop the row which look like this, they are the info from the scanned files
result1 = result1.dropna(subset=['Business Name', 'Jurisdiction', 'Registered Office Name', 
                'Registered Office Address', 'Principal Office Address', 'Business'],
              how='all')
result2 = result2.dropna(subset=['Business Name', 'Jurisdiction', 'Registered Office Name', 
                'Registered Office Address', 'Principal Office Address', 'Business'],
              how='all')

In [ ]:
# sometimes the info of the registered office address and name is sticked together, I separate them here
result1['Registered Office Name'] = result1['Registered Office Address'].str.extract(r'(.*?),')
result1['Registered Office Address'] = result1['Registered Office Address'].str.extract(r', (.*)')

result2['Registered Office Name'] = result2['Registered Office Address'].str.extract(r'(.*?),')
result2['Registered Office Address'] = result2['Registered Office Address'].str.extract(r', (.*)')